In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Nitin\\TEXT-SUMMARISER\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Nitin\\TEXT-SUMMARISER'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            eval_strategy = params.eval_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\Nitin\TEXT-SUMMARISER\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\Nitin\TEXT-SUMMARISER\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # trainer_args = TrainingArguments(
        #     output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
        #     per_device_train_batch_size=self.config.per_device_train_batch_size, per_device_eval_batch_size=self.config.per_device_train_batch_size,
        #     weight_decay=self.config.weight_decay, logging_steps=self.config.logging_steps,
        #     evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,
        #     gradient_accumulation_steps=self.config.gradient_accumulation_steps
        # ) 


        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=1, warmup_steps=500,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            weight_decay=0.01, logging_steps=10,
            eval_strategy='steps', eval_steps=500, save_steps=1e6,
            gradient_accumulation_steps=16
        ) 

        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"], 
                  eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))

In [10]:
import os
print(os.path.exists("config/config.yaml"))
print(os.listdir("."))  # shows all files/folders in current directory

True
['.git', '.github', '.gitignore', 'app.py', 'artifacts', 'config', 'Dockerfile', 'LICENSE', 'logs', 'main.py', 'params.yaml', 'README.md', 'requirements.txt', 'research', 'setup.py', 'src', 'srctextSummarizercomponents__init__.py', 'template.py', 'venv']


In [11]:
import os
print(os.getcwd())
print(os.listdir("."))

c:\Users\Nitin\TEXT-SUMMARISER
['.git', '.github', '.gitignore', 'app.py', 'artifacts', 'config', 'Dockerfile', 'LICENSE', 'logs', 'main.py', 'params.yaml', 'README.md', 'requirements.txt', 'research', 'setup.py', 'src', 'srctextSummarizercomponents__init__.py', 'template.py', 'venv']


In [12]:
# Search for config.yaml on your system
for root, dirs, files in os.walk("c:/Users/Nitin/TEXT-SUMMARISER"):
    for file in files:
        if file == "config.yaml":
            print(os.path.join(root, file))

c:/Users/Nitin/TEXT-SUMMARISER\config\config.yaml


In [13]:
import os
os.chdir("c:/Users/Nitin/TEXT-SUMMARISER")
print(os.getcwd())
print(os.path.exists("config/config.yaml"))

c:\Users\Nitin\TEXT-SUMMARISER
True


In [14]:
import os
for root, dirs, files in os.walk("c:/Users/Nitin/TEXT-SUMMARISER"):
    # skip venv folder
    dirs[:] = [d for d in dirs if d != 'venv']
    for file in files:
        if file.endswith((".py", ".yaml", ".yml", ".ipynb")):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f, 1):
                    if "evaluation_strategy" in line:
                        print(f"FOUND → {filepath} | Line {i}: {line.strip()}")

print("Done.")

FOUND → c:/Users/Nitin/TEXT-SUMMARISER\research\04_model_trainer.ipynb | Line 198: "        #     evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,\n",
FOUND → c:/Users/Nitin/TEXT-SUMMARISER\research\04_model_trainer.ipynb | Line 328: "FOUND → c:/Users/Nitin/TEXT-SUMMARISER\\research\\04_model_trainer.ipynb | Line 198: \"        #     evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,\\n\",\n",
FOUND → c:/Users/Nitin/TEXT-SUMMARISER\research\04_model_trainer.ipynb | Line 329: "FOUND → c:/Users/Nitin/TEXT-SUMMARISER\\research\\04_model_trainer.ipynb | Line 331: \"FOUND → c:/Users/Nitin/TEXT-SUMMARISER\\\\research\\\\04_model_trainer.ipynb | Line 187: \\\"        #     evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,\\\\n\\\",\\n\",\n",
FOUND → c:/Users/Nitin/TEXT-SUMMARISER\research\04_model_trainer.ipynb | Line 330: "FOUND → c

In [15]:
import json

with open("research/04_model_trainer.ipynb", "r", encoding="utf-8") as f:
    content = f.read()

# Fix all occurrences
content = content.replace(
    "eval_strategy = params.eval_strategy",
    "eval_strategy = params.eval_strategy"
).replace(
    "eval_steps = params.eval_steps",
    "eval_steps = params.eval_steps"
).replace(
    "eval_strategy: str",
    "eval_strategy: str"
).replace(
    "eval_strategy='steps'",
    "eval_strategy='steps'"
)

with open("research/04_model_trainer.ipynb", "w", encoding="utf-8") as f:
    f.write(content)

print("Done. Now close and reopen the notebook.")

Done. Now close and reopen the notebook.


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-03-23 00:41:51,155: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-23 00:41:51,165: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-23 00:41:51,169: INFO: common: created directory at: artifacts]
[2026-03-23 00:41:51,171: INFO: common: created directory at: artifacts/model_trainer]
